# **Car Price Predcition**

### **Importing Libraries**



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder



## **Data Pre-processing**

In [5]:
dataset = pd.read_csv('car_details.csv')

In [6]:
dataset.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [7]:
dataset.shape

(8128, 13)

In [8]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           8128 non-null   object 
 1   year           8128 non-null   int64  
 2   selling_price  8128 non-null   int64  
 3   km_driven      8128 non-null   int64  
 4   fuel           8128 non-null   object 
 5   seller_type    8128 non-null   object 
 6   transmission   8128 non-null   object 
 7   owner          8128 non-null   object 
 8   mileage        7907 non-null   object 
 9   engine         7907 non-null   object 
 10  max_power      7913 non-null   object 
 11  torque         7906 non-null   object 
 12  seats          7907 non-null   float64
dtypes: float64(1), int64(3), object(9)
memory usage: 825.6+ KB


In [9]:
dataset.isnull().sum()

,0
name,0
year,0
selling_price,0
km_driven,0
fuel,0
seller_type,0
transmission,0
owner,0
mileage,221
engine,221


In [10]:
print(dataset.fuel.value_counts())
print(dataset.transmission.value_counts())
print(dataset.owner.value_counts())

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64
transmission
Manual       7078
Automatic    1050
Name: count, dtype: int64
owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64


In [11]:
dataset = dataset.dropna()

In [12]:
dataset.isnull().sum()

,0
name,0
year,0
selling_price,0
km_driven,0
fuel,0
seller_type,0
transmission,0
owner,0
mileage,0
engine,0


In [13]:
dataset = dataset[dataset['owner'] != 'Test Drive Car']

In [14]:

dataset = dataset.drop(['torque'], axis=1)

In [15]:

dataset['mileage'] = dataset['mileage'].str.extract('(\d+\.\d+|\d+)').astype(float)
dataset['engine'] = dataset['engine'].str.extract('(\d+\.\d+|\d+)').astype(float)
dataset['max_power'] = dataset['max_power'].str.extract('(\d+\.\d+|\d+)').astype(float)

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_7798/2951342326.py:1: SyntaxWarning: invalid escape sequence '\d'
  dataset['mileage'] = dataset['mileage'].str.extract('(\d+\.\d+|\d+)').astype(float)
/tmp/ipykernel_7798/2951342326.py:2: SyntaxWarning: invalid escape sequence '\d'
  dataset['engine'] = dataset['engine'].str.extract('(\d+\.\d+|\d+)').astype(float)
/tmp/ipykernel_7798/2951342326.py:3: SyntaxWarning: invalid escape sequence '\d'
  dataset['max_power'] = dataset['max_power'].str.extract('(\d+\.\d+|\d+)').astype(float)


In [16]:

dataset = dataset.reset_index(drop=True)

In [17]:
dataset.shape

(7901, 12)

In [18]:

dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7901 entries, 0 to 7900
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           7901 non-null   object 
 1   year           7901 non-null   int64  
 2   selling_price  7901 non-null   int64  
 3   km_driven      7901 non-null   int64  
 4   fuel           7901 non-null   object 
 5   seller_type    7901 non-null   object 
 6   transmission   7901 non-null   object 
 7   owner          7901 non-null   object 
 8   mileage        7901 non-null   float64
 9   engine         7901 non-null   float64
 10  max_power      7901 non-null   float64
 11  seats          7901 non-null   float64
dtypes: float64(4), int64(3), object(5)
memory usage: 740.8+ KB


In [19]:
dataset.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248.0,74.00,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498.0,103.52,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497.0,78.00,5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396.0,90.00,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298.0,88.20,5.0


 ## Splitting the Data

In [20]:
x = dataset.drop('selling_price', axis=1)

In [21]:
x

,name,year,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
0,Maruti Swift Dzire VDI,2014,145500,Diesel,Individual,Manual,First Owner,23.40,1248.0,74.00,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,120000,Diesel,Individual,Manual,Second Owner,21.14,1498.0,103.52,5.0
2,Honda City 2017-2020 EXi,2006,140000,Petrol,Individual,Manual,Third Owner,17.70,1497.0,78.00,5.0
3,Hyundai i20 Sportz Diesel,2010,127000,Diesel,Individual,Manual,First Owner,23.00,1396.0,90.00,5.0
4,Maruti Swift VXI BSIII,2007,120000,Petrol,Individual,Manual,First Owner,16.10,1298.0,88.20,5.0
...,...,...,...,...,...,...,...,...,...,...,...
7896,Hyundai i20 Magna,2013,110000,Petrol,Individual,Manual,First Owner,18.50,1197.0,82.85,5.0
7897,Hyundai Verna CRDi SX,2007,119000,Diesel,Individual,Manual,Fourth & Above Owner,16.80,1493.0,110.00,5.0
7898,Maruti Swift Dzire ZDi,2009,120000,Diesel,Individual,Manual,First Owner,19.30,1248.0,73.90,5.0
7899,Tata Indigo CR4,2013,25000,Diesel,Individual,Manual,First Owner,23.57,1396.0,70.00,5.0


In [22]:
y = dataset['selling_price']

In [23]:
y

,selling_price
0,450000
1,370000
2,158000
3,225000
4,130000
...,...
7896,320000
7897,135000
7898,382000
7899,290000


## **Label Encoding**

In [24]:
le = LabelEncoder()


In [25]:

le = LabelEncoder()
x['name'] = le.fit_transform(x['name'])
x['fuel'] = le.fit_transform(x['fuel'])
x['seller_type'] = le.fit_transform(x['seller_type'])
x['transmission'] = le.fit_transform(x['transmission'])
x['owner'] = le.fit_transform(x['owner'])



## **Train-Test Split**

In [26]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [27]:
print("x_train shape:", x_train.shape)

x_train shape: (6320, 11)


In [28]:
print("x_test shape:", x_test.shape)


x_test shape: (1581, 11)


# **Random Forest**

In [29]:
rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)

In [30]:


rf_params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5]
}

In [31]:
rf_model =RandomForestRegressor(random_state=42, n_jobs=-1)

In [32]:
rf_search = RandomizedSearchCV(rf_model, rf_params, cv=5, scoring='r2', n_iter=15, verbose=2, n_jobs=-1)

In [33]:
rf_search.fit(x_train, y_train)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


RandomizedSearchCV(cv=5,
                   estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'max_depth': [None, 10, 20, 30],
                                        'max_features': ['sqrt', 'log2', 0.5],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   scoring='r2', verbose=2)

In [34]:
rf_preds = rf_search.predict(x_test)


In [35]:
print("Best Parameters Found:", rf_search.best_params_)

Best Parameters Found: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': None}


In [36]:
print("Best CV R2 Score:", round(rf_search.best_score_, 4))

Best CV R2 Score: 0.9673


In [37]:
print("R2 Score:", round(r2_score(y_test, rf_preds), 4))

print("MAE (Average Rupee Error):", round(mean_absolute_error(y_test, rf_preds), 2))

print("RMSE:", round(np.sqrt(mean_squared_error(y_test, rf_preds)), 2))

R2 Score: 0.9774
MAE (Average Rupee Error): 59691.73
RMSE: 118801.89


# **XG-Boost**

In [51]:
xgb_params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2]
}

In [52]:
xgb_model =xgb.XGBRegressor(tree_method='hist',random_stae=42)

In [53]:
xgb_search = RandomizedSearchCV(xgb_model, xgb_params, cv=5, scoring='r2', n_iter=15, verbose=2, n_jobs=-1)


In [54]:
xgb_search.fit(x_train, y_train)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:35:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "random_stae" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                                          min_child_weight=None, missing=nan,
                                          monotone_constraints=None,
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None, ...),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9,
                                                             1.0],
                                        'gamma': [0, 0.1, 0.2],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.2],
                                        'max_depth': [3, 5, 7, 9],
                                        'n_estimators': [100, 200, 300, 500],
                                        'subsample': [0.7, 0.8, 0.9, 1.0]},
                   scoring='r2', verbose=2)

In [43]:
xgb_preds  = xgb_search.predict(x_test)

In [44]:
print("Best Parameters Found:", xgb_search.best_params_)

Best Parameters Found: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.2, 'gamma': 0, 'colsample_bytree': 0.9}


In [45]:
print("Best CV  R2 Score :" , round(xgb_search.best_score_,4))

Best CV  R2 Score : 0.9694


In [46]:
print("R2 Score:", round(r2_score(y_test, xgb_preds), 4))


R2 Score: 0.9799


In [47]:
print("MAE (Average Rupee Error):",round(mean_absolute_error(y_test,xgb_preds),2))

MAE (Average Rupee Error): 63199.82


In [48]:
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, xgb_preds)), 2))

RMSE: 111989.16


## **Optuna**

In [72]:
import optuna



def objective(trial):


    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10)
    }


    model = xgb.XGBRegressor(**params, tree_method='hist', random_state=42)


    from sklearn.model_selection import cross_val_score
    scores = cross_val_score(model, x_train, y_train, cv=5, scoring='r2')


    return scores.mean()


#  'maximize' because we want the highest R2 score
study = optuna.create_study(direction='maximize')

print("Starting Optuna... let it run for 3 minutes!")
study.optimize(objective, timeout=180)

print("--- OPTUNA RESULTS ---")
print("Best R2 Score found:", round(study.best_value, 4))
print("Best Parameters:", study.best_params)

[I 2026-06-06 08:33:15,716] A new study created in memory with name: no-name-7c277693-19f8-4461-ae3b-292e0b0eb35c


Starting Optuna... let it run for 3 minutes!


[I 2026-06-06 08:33:19,012] Trial 0 finished with value: 0.9560717701911926 and parameters: {'n_estimators': 285, 'max_depth': 3, 'learning_rate': 0.06691017890072756, 'subsample': 0.6605993635599043, 'colsample_bytree': 0.5106379308514397, 'gamma': 0.3390030589690631, 'reg_alpha': 9.812069567422675, 'reg_lambda': 9.521216503904157}. Best is trial 0 with value: 0.9560717701911926.
[I 2026-06-06 08:33:26,959] Trial 1 finished with value: 0.9653617739677429 and parameters: {'n_estimators': 588, 'max_depth': 7, 'learning_rate': 0.21615149642931636, 'subsample': 0.7897232010033317, 'colsample_bytree': 0.6772870563912754, 'gamma': 0.20998784767400996, 'reg_alpha': 3.7369670041308103, 'reg_lambda': 4.048266271903884}. Best is trial 1 with value: 0.9653617739677429.
[I 2026-06-06 08:33:34,266] Trial 2 finished with value: 0.9674617290496826 and parameters: {'n_estimators': 818, 'max_depth': 7, 'learning_rate': 0.07941962620289533, 'subsample': 0.8449384153889894, 'colsample_bytree': 0.7624179


--- OPTUNA RESULTS ---
Best R2 Score found: 0.9697
Best Parameters: {'n_estimators': 849, 'max_depth': 5, 'learning_rate': 0.13446744940105856, 'subsample': 0.9714241527938928, 'colsample_bytree': 0.896500151805599, 'gamma': 0.29099868484722446, 'reg_alpha': 8.73033377946944, 'reg_lambda': 1.5279478108713445}


In [73]:

best_params = study.best_params
final_optuna_xgb = xgb.XGBRegressor(**best_params, tree_method='hist', random_state=42)


final_optuna_xgb.fit(x_train, y_train)
optuna_preds = final_optuna_xgb.predict(x_test)


optuna_r2 = round(r2_score(y_test, optuna_preds), 4)
optuna_mae = round(mean_absolute_error(y_test, optuna_preds), 2)
optuna_rmse = round(np.sqrt(mean_squared_error(y_test, optuna_preds)), 2)

print("--- OPTUNA XGBOOST METRICS ---")
print("R2:", optuna_r2)
print("MAE:", optuna_mae)
print("RMSE:", optuna_rmse)

--- OPTUNA XGBOOST METRICS ---
R2: 0.9812
MAE: 56405.55
RMSE: 108372.7


# **FINAL MODEL COMPARISON**

In [75]:

rf_r2, rf_mae, rf_rmse = 0.9774, 59691.73, 118801.89
xgb_r2, xgb_mae, xgb_rmse = 0.9799, 63199.82, 111989.16



print(f"{'Metric':<8} | {'Random Forest':<14} | {'Basic XGB':<14} | {'Optuna XGB':<14}")


print(f"{'R2':<8} | {rf_r2:<14} | {xgb_r2:<14} | {optuna_r2:<14}")
print(f"{'MAE':<8} | {rf_mae:<14} | {xgb_mae:<14} | {optuna_mae:<14}")
print(f"{'RMSE':<8} | {rf_rmse:<14} | {xgb_rmse:<14} | {optuna_rmse:<14}")



Metric   | Random Forest  | Basic XGB      | Optuna XGB    
R2       | 0.9774         | 0.9799         | 0.9812        
MAE      | 59691.73       | 63199.82       | 56405.55      
RMSE     | 118801.89      | 111989.16      | 108372.7      
